In [1]:
import whisper
import time
import os

# Load the base model (good balance of speed and accuracy for testing)
# In production, you might use "small", "medium", or "large"
print("Loading Whisper model...")
model = whisper.load_model("base")
print("Model loaded successfully!")

Loading Whisper model...
Model loaded successfully!


In [2]:
# Path to your test audio file
audio_file_path = "../data/uploads/harvard.wav"

print(f"Transcribing {audio_file_path}...")
start_time = time.time()

# Run the transcription
result = model.transcribe(audio_file_path)

end_time = time.time()
print(f"Transcription complete in {round(end_time - start_time, 2)} seconds.\n")

# Print the raw text
transcript_text = result["text"]
print("--- TRANSCRIPT ---")
print(transcript_text)

Transcribing ../data/uploads/harvard.wav...


/Users/sarveshsharma/Downloads/Projects/meeting-intelligence-pipeline/project1/lib/python3.9/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription complete in 0.88 seconds.

--- TRANSCRIPT ---
 The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.


In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser

# Import our exact schemas from Phase 1
import sys
sys.path.append("..") # Allows us to import from the backend folder
from backend.models.schemas import MeetingSummary

# Load your OpenAI API key from the .env file
load_dotenv("../.env")

/Users/sarveshsharma/Downloads/Projects/meeting-intelligence-pipeline/project1/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


True

In [5]:
# Initialize the parser with our strict Pydantic class
parser = PydanticOutputParser(pydantic_object=MeetingSummary)

# Create the prompt template
prompt = PromptTemplate(
    template="""
    You are an expert executive assistant. Review the following meeting transcript and extract the key information.
    
    {format_instructions}
    
    MEETING TRANSCRIPT:
    {transcript}
    """,
    input_variables=["transcript"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# Initialize the LLM (GPT-4o-mini is fast and cheap for testing)
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

# Combine the prompt and the LLM into a chain
chain = prompt | llm | parser

In [6]:
# Paste the text you got from the Whisper notebook here
sample_transcript = """
Hey everyone, thanks for joining. Let's keep this quick. 
Sarah, the client loved the new designs, but they want the dashboard color changed to blue. 
Can you get that updated by next Tuesday? 
Also, we decided to push the marketing launch back to November to give us more time. 
Overall, a really great meeting with them, they seemed very happy.
"""

print("Extracting intelligence...")
# Run the chain!
structured_output = chain.invoke({"transcript": sample_transcript})

# Print the beautiful, structured JSON
print(structured_output.model_dump_json(indent=2))

Extracting intelligence...
{
  "executive_summary": "In the meeting, the team discussed feedback from the client regarding the new designs. The client expressed satisfaction but requested a change in the dashboard color to blue. Additionally, the team made a decision to postpone the marketing launch to November to allow for more preparation time. Overall, the meeting was positive, with the client appearing very happy with the progress.",
  "key_decisions": [
    "Change the dashboard color to blue as requested by the client.",
    "Push the marketing launch back to November."
  ],
  "action_items": [
    {
      "task": "Update the dashboard color to blue.",
      "assignee": "Sarah",
      "due_date": "Next Tuesday",
      "priority": "Normal"
    }
  ],
  "overall_sentiment": "Positive"
}
